# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [18]:
# LANE: Lane 4 — CTR / Engagement Opportunity Scoring (Core Lane)
# WHY: SEOs often treat all pages the same when assessing click-through rates (CTR).
# However, as per Lane 4 guidelines, different content types and position tiers have completely 
# different baseline curves. By mapping content metadata with position_tier and competition_level,
# we can score which pages are under-capturing clicks relative to their specific cohort.

import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Mean CTR by Content Type:")
for c_type, val in df.groupby("content_type")["ctr"].mean().items():
    print(f" - {c_type}: {val:.4f}%")

Mean CTR by Content Type:
 - comparison article: 0.1312%
 - feedly article: 2.7913%
 - keyword article: 0.3448%


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [14]:
# DECISION: Which specific pages content editors should spend their limited hours rewriting or optimizing first.
# ACTION: Content writers and SEO managers will revise the low-performing content.
# COST OF A WRONG CALL: A false positive (flagging a healthy page) wastes hours of an editor's time rewriting
# a page that won't see any traffic lift. A false negative (missing a broken page) means we continue to leak traffic.
# WHY ML: A fixed rule like "flag everything with CTR < 1%" fails because content types, position tiers, 
# and keyword competition levels have completely different baseline CTR curves. ML untangles these interactions.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [15]:
# 1. Total viable pages for analysis (impressions >= 100)
# 2. Minimum vs Maximum average CTR by Content Type
# 3. Total underperforming pages available for prioritization queue

import numpy as np

# FIX: Use content-type specific medians instead of a global median to avoid penalizing inherently lower-CTR groups.
content_medians = df[df["impressions_90d"] >= 100].groupby("content_type")["ctr"].median()
df["median_ctr_threshold"] = df["content_type"].map(content_medians)

df["is_underperforming"] = np.where(
    (df["impressions_90d"] >= 100) & (df["ctr"] < df["median_ctr_threshold"]), 1, 0
)

viable_count = (df["impressions_90d"] >= 100).sum()
type_ctrs = df[df["impressions_90d"] >= 100].groupby("content_type")["ctr"].mean()
underperforming_count = df["is_underperforming"].sum()

print(f"1. Total high-visibility pages to analyze: {viable_count:,}")
print(f"2. CTR baseline difference: '{type_ctrs.idxmin()}' average CTR is {type_ctrs.min():.4f}%, "
      f"while '{type_ctrs.idxmax()}' is {type_ctrs.max():.4f}%.")
print(f"3. Total underperforming pages found (relative to their content type benchmark): {underperforming_count:,}")

1. Total high-visibility pages to analyze: 22,006
2. CTR baseline difference: 'comparison article' average CTR is 0.1340%, while 'feedly article' is 0.7061%.
3. Total underperforming pages found (relative to their content type benchmark): 10,753


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [16]:
# CAN CLAIM (Decision-Support / Directional): "Based on content metadata and historical patterns, 
# Page A is directionally more likely to be underperforming than Page B and should be audited first."
# LEAKAGE PREVENTION: We are omitting 'trend_direction' and 'trend_pct' from features as they are the outcome in disguise.

# NOTE ON DATA HYGIENE (CRITICAL FIX FOR THE ZERO-TRAP): 
# Per the reviewer's note and data dictionary, 'avg_position = 0' signifies "no data," not rank zero.
# In our pipeline step, we will treat 0 as a structural missing value (NaN) rather than a numerical minimum.

safe_features = [
    "content_type", "main_intent", "competition_level", 
    "position_tier", # representation of rank range, safe from leakage
    "impressions_90d", "sessions_90d", 
    "content_age_days", "days_since_last_update"
]
print("Safe Model Input Features:", safe_features)

Safe Model Input Features: ['content_type', 'main_intent', 'competition_level', 'position_tier', 'impressions_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.